# 🏠 German Rental Price Classification

**Dataset:** `immo_data.csv` (German rental listings scraped from ImmoScout24)
**Task:** Multi-class classification
**Method:** CRISP-DM (see `PROJECT_ROADMAP.md`)

---

## 1. Business Understanding

### 1.1 Background
Rent prices in Germany vary widely between regions, cities and types of property. A tenant browsing hundreds of listings finds it hard to judge whether a specific offer is **cheap, fair, or expensive** for what it offers, and a rental platform has no automatic way to tell them.

### 1.2 Problem (draft)
- **Main user:** a rental platform (e.g., ImmoScout24) and, through it, the tenants who use it.
- **What the model does:** predict the **expected** rent class (Low / Medium / High per m²) of a flat from its attributes (location, size, age, condition, equipment…), **without seeing its price**.
- **Decision it supports:** the platform compares the predicted class with the flat's **actual** price:
  - Predicted *Medium*, actually *High* → ⚠️ **possibly overpriced**
  - Predicted *High*, actually *Medium* → 💎 **good deal**

  This helps tenants judge whether an offer is fair.
- **Secondary goal:** find **which attributes drive** a flat into a higher price class.

### 1.3 Target definition
- **Target (decision attribute):** `rentClass` ∈ {Low, Medium, High}
- **Built from:** `rentPerSqm = baseRent ÷ livingSpace`
- **Why per m² and not total rent?** Total rent mainly reflects the flat's **size**. Rent per m² removes the size effect and shows the true **price level**, so small and large flats can be compared fairly.

  | Flat | Size | Total rent | Rent per m² |
  |---|---|---|---|
  | A | 30 m² | €600 | **€20/m²** (expensive) |
  | B | 120 m² | €1,200 | **€10/m²** (cheaper) |

- **How the classes are made:** sort all flats by rent per m² and cut the list into **3 equal-sized groups** (tertiles). Exact cut-off values will be computed in Phase 5.
- **Leakage warning:** since the target is built from rent, every rent-related column (`baseRent`, `totalRent`, `baseRentRange`, `serviceCharge`…) must be **removed from the model inputs**. Otherwise the model would be reading the answer.

### 1.4 Success criteria (draft, to be verified in Phase 5.4)
| Item | Value | Reasoning |
|---|---|---|
| **Assumption** | Classes ≈ balanced (~33% each) | The classes are created by splitting into 3 equal groups |
| **Baseline** | ≈ 33% accuracy | A "no-brain" model that always predicts one class gets 1 in 3 right |
| **Main goal** | **Accuracy ≥ 70%** | Clearly beats the baseline; "good" range for a balanced 3-class problem |
| **Mistake priority** | **Far mistakes (Low ↔ High) must be rare** | The classes are *ordered*: a near mistake (High↔Medium) gives a mild misjudgement, but a far mistake (High↔Low) would put a strongly wrong "overpriced" or "good deal" label on a flat and mislead tenants |
| **Also reported** | Precision / recall / F1 per class, balanced accuracy, confusion matrix | To make sure no single class is predicted badly |

> **Why accuracy is acceptable here:** it is a valid main measure only when (1) the classes are balanced **and** (2) mistakes cost about the same. (1) is true by design. (2) is only partly true, so we add a confusion-matrix check on far mistakes.
>
> If Phase 5.4 shows the classes are *not* balanced, switch the main goal to **balanced accuracy / macro F1**.


In [1]:
print(sys.executable)

NameError: name 'sys' is not defined